
http://localhost:4040/jobs/

In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, collect_list

import time


def create_spark_session() -> SparkSession:
    conf = SparkConf().set("spark.driver.memory", "8g")

    spark_session = SparkSession\
        .builder\
        .master("local[4]")\
        .config(conf=conf)\
        .appName("Spark UI Tutorial") \
        .getOrCreate()

    return spark_session


In [2]:
spark = create_spark_session()

In [3]:
test_df = spark.createDataFrame([
    (1, 'a'),
    (2, 'b'),
    (3, 'c'),
    (4, 'd'),
    (5, 'e'),
    (6, 'f'),
    (7, 'g'),
    (8, 'h'),
    (9, 'i'),
    (10, 'j')
], ["number", "letter"]).cache()

In [9]:
test_df.show(truncate=False)

+------+------+
|number|letter|
+------+------+
|1     |a     |
|2     |b     |
|3     |c     |
|4     |d     |
|5     |e     |
|6     |f     |
|7     |g     |
|8     |h     |
|9     |i     |
|10    |j     |
+------+------+



In [14]:
test_df \
    .withColumn("mod", col("number") % 2) \
    .groupBy("mod") \
    .agg(collect_list("letter").alias("letter")) \
    .show(truncate=False)

# For UI to stick
time.sleep(100)

+---+---------------+
|mod|letter         |
+---+---------------+
|0  |[b, d, f, h, j]|
|1  |[a, c, e, g, i]|
+---+---------------+



In [4]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [5]:
sql_create_database = """
create database if not exists analytics
location './user/cloudera/analytics/'
"""
result_create_db = spark.sql(sql_create_database)

In [6]:
test_df.createOrReplaceTempView("test_data")

In [ ]:
sql_create_table = """ 
create table if not exists analytics.pandas_spark_hive 
using parquet as select 
to_timestamp(date) as date_parsed, * 
from test_data
"""
result_create_table = spark.sql(sql_create_table)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `date` cannot be resolved. Did you mean one of the following? [`letter`, `number`].; line 4 pos 13;
'CreateTable `spark_catalog`.`analytics`.`pandas_spark_hive`, Ignore
+- 'Project ['to_timestamp('date) AS date_parsed#25, number#0L, letter#1]
   +- SubqueryAlias test_data
      +- View (`test_data`, [number#0L,letter#1])
         +- LogicalRDD [number#0L, letter#1], false


In [11]:
spark.stop()